# PySpark SQL — Data Loading, Transformation & Querying

This notebook walks through a complete Spark SQL workflow:
- Setting up a SparkSession with Hive support
- Reading structured CSV data with custom schemas
- Registering temp views for SQL queries
- Performing joins, transformations, and saving results as managed tables

In [ ]:
# Step 1: Import SparkSession
# SparkSession is the main entry point for Spark functionality.
# It replaces the older SQLContext and HiveContext.
from pyspark.sql import SparkSession

In [ ]:
# Step 2: Create and configure the Spark session
spark = (
    SparkSession
    .builder
    .appName("HR_Data_Pipeline")
    .master("local[*]")
    .enableHiveSupport()
    .config("spark.sql.warehouse.dir", "/data/output/hive-warehouse")
    .getOrCreate()
)

# Configuration notes:
# appName          -> Identifies this job in the Spark UI
# local[*]         -> Uses all available CPU cores on the local machine
# enableHiveSupport-> Enables Hive metastore for table persistence
# warehouse.dir    -> Directory where managed table data is stored
# getOrCreate      -> Returns existing session if one already exists

In [ ]:
# Step 3: Define schema for the staff records CSV
# Providing a schema upfront avoids the overhead of Spark's schema inference
staff_schema = """
    first_name     string,
    last_name      string,
    designation    string,
    date_of_birth  string,
    email_address  string,
    contact_number string,
    monthly_salary double,
    dept_code      int
"""

In [ ]:
# Step 4: Load staff CSV into a DataFrame
staff_df = (
    spark.read
    .format("csv")
    .schema(staff_schema)
    .option("header", True)
    .load("/data/input/staff_records.csv")
)

# format("csv")      -> Tells Spark to read a CSV file
# schema(...)        -> Uses our pre-defined column types (avoids slow inference)
# option("header")   -> Skips the first row since it contains column names
# load(path)         -> Points to the actual file location on disk

print(f"Rows loaded: {staff_df.count()}")
staff_df.printSchema()

In [ ]:
# Step 5: Define schema for department reference data
dept_schema = """
    dept_code        int,
    dept_name        string,
    dept_description string,
    location_city    string,
    location_state   string,
    country          string
"""

In [ ]:
# Step 6: Load department CSV
dept_df = (
    spark.read
    .format("csv")
    .schema(dept_schema)
    .option("header", True)
    .load("/data/input/dept_master.csv")
)

print(f"Departments loaded: {dept_df.count()}")
dept_df.show(5)

In [ ]:
# Step 7: Check which catalog (metastore) Spark is using
catalog_type = spark.conf.get("spark.sql.catalogImplementation")
print(f"Catalog in use: {catalog_type}")

# Possible values:
# 'hive'      -> Persistent Hive metastore (tables survive session restarts)
# 'in-memory' -> Tables are lost when the session ends

In [ ]:
# Step 8: List all available databases in the metastore
spark.sql("SHOW DATABASES").show()

# Step 9: Check which tables already exist in the default database
spark.sql("SHOW TABLES IN default").show()

In [ ]:
# Step 10: Register DataFrames as temporary SQL views
# Temp views exist only for the current session — they are not saved to disk
staff_df.createOrReplaceTempView("staff_view")
dept_df.createOrReplaceTempView("dept_view")

print("Temp views registered: staff_view, dept_view")

In [ ]:
# Step 11: Filter staff belonging to a specific department using SQL
dept_filter_df = spark.sql("""
    SELECT *
    FROM staff_view
    WHERE dept_code = 3
""")

print(f"Employees in dept 3: {dept_filter_df.count()}")
dept_filter_df.show()

In [ ]:
# Step 12: Extract birth year from date_of_birth for demographic analysis
staff_with_year = spark.sql("""
    SELECT
        *,
        date_format(date_of_birth, 'yyyy') AS birth_year
    FROM staff_view
""")

# date_format() formats a date column using a Java SimpleDateFormat pattern
# 'yyyy' extracts only the 4-digit year component
staff_with_year.show(5)

In [ ]:
# Step 13: Register the enriched DataFrame as another temp view
staff_with_year.createOrReplaceTempView("staff_enriched_view")

# Verify the new view is accessible
spark.sql("SELECT first_name, designation, birth_year FROM staff_enriched_view LIMIT 5").show()

In [ ]:
# Step 14: Join staff with department data using SQL + broadcast hint
# BROADCAST hint tells Spark to replicate the smaller dept table to all executors
# This avoids a costly shuffle when joining a large and small table

final_df = spark.sql("""
    SELECT /*+ BROADCAST(d) */
        s.*,
        d.dept_name,
        d.location_city
    FROM staff_view s
    LEFT OUTER JOIN dept_view d
        ON s.dept_code = d.dept_code
""")

# LEFT OUTER JOIN -> keeps all staff records even if dept info is missing
print(f"Final joined records: {final_df.count()}")
final_df.show(10, truncate=False)

In [ ]:
# Step 15: Persist final DataFrame as a managed Spark table in Parquet format
final_df.write.format("parquet").saveAsTable("staff_dept_combined")

# Why Parquet?
# -> Columnar storage: reads only required columns (faster queries)
# -> Built-in compression: smaller storage footprint
# -> Native support in Hive, Presto, Athena, BigQuery
# -> Schema embedded in file (no need for external schema management)

print("Table saved: staff_dept_combined")

In [ ]:
# Step 16: Query the saved table to verify data was stored correctly
verification_df = spark.sql("""
    SELECT dept_name, COUNT(*) AS headcount, ROUND(AVG(monthly_salary), 2) AS avg_salary
    FROM staff_dept_combined
    GROUP BY dept_name
    ORDER BY headcount DESC
""")

verification_df.show()

In [ ]:
# Step 17: Inspect the saved table's metadata
spark.sql("DESCRIBE EXTENDED staff_dept_combined").show(50, truncate=False)

# DESCRIBE EXTENDED shows:
# - Column names and data types
# - Storage format (parquet here)
# - Table owner and creation time
# - HDFS/warehouse location of table files
# - Row count statistics (if collected)

## Pipeline Overview

```
SparkSession (with Hive)
        |
   Read CSV Files
   (staff + dept)
        |
 Create Temp Views
        |
 SQL Queries & Filters
        |
  Date Extraction
        |
 LEFT JOIN + BROADCAST
        |
 Save as Parquet Table
        |
 Read Back & Describe
```

| Concept | What it does |
| --- | --- |
| `createOrReplaceTempView` | Makes a DataFrame queryable via SQL within the session |
| `BROADCAST` hint | Sends small table to all nodes to avoid shuffle joins |
| `LEFT OUTER JOIN` | Keeps all left-side rows even without a match |
| `saveAsTable` | Saves as a persistent managed table (Parquet format) |
| `DESCRIBE EXTENDED` | Shows table schema + storage metadata |